<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [1]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

📂 Montando Google Drive...
Mounted at /content/drive
📍 Banco de Dados: /content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega/data/base-dados.db


In [2]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 28.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
🛠️ Verificando integridade das tabelas...
✅ Estrutura (Schema) validada com sucesso!

🚀 Ambiente pronto (GPU: Ativa).


In [3]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.
✅ Carga de /content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega/sql/02-seed_data.sql concluída com sucesso!


In [4]:
# Célula 4: Indexação Hierárquica e Sensores XAI (Versão Protegida contra Locks)
import spacy
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Preparação do Modelo NLP
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def executar_indexacao_completa(db_path):
    # Inicializamos a conexão como None para o bloco finally reconhecê-la
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # --- PREPARAÇÃO DO BANCO ---
        print("🧹 Limpando índices anteriores...")
        cursor.execute("DELETE FROM verso_palavra")
        cursor.execute("DELETE FROM palavra")
        cursor.execute("DELETE FROM verso_limpo")

        # 2. Carga dos Versos (Ajustado para Filipenses conforme solicitado)
        query = """
            SELECT v.id, v.texto
            FROM verso v
            JOIN livro l ON v.livro_id = l.id
            WHERE v.processar = 'S'
        """
        df_versos = pd.read_sql_query(query, conn)

        print(f"🧠 Analisando {len(df_versos)} versos...")

        for _, row in tqdm(df_versos.iterrows(), total=len(df_versos), desc="Gramática e Sensores"):
            verso_id = row['id']
            texto = row['texto']

            if not texto or len(texto.strip()) < 3:
                continue

            doc = nlp(texto)
            total_tokens = len(doc)

            # Acumuladores de Metadados
            counts = {'ADJ': 0, 'ADV': 0, 'PROPN': 0, 'VERB': 0, 'NUM': 0, 'NOUN': 0}
            n_primeira_pessoa = 0
            palavras_sig_len = []
            is_identidade = 0
            tem_numeral = 0

            # --- PROCESSAMENTO POR TOKEN ---
            for t in doc:
                # Gestão do Dicionário 'palavra'
                lemma_lower = t.lemma_.lower()
                cursor.execute("""
                    INSERT OR IGNORE INTO palavra (lemma, pos_tag, is_stop)
                    VALUES (?, ?, ?)
                """, (lemma_lower, t.pos_, 1 if t.is_stop else 0))

                cursor.execute("SELECT id FROM palavra WHERE lemma = ?", (lemma_lower,))
                palavra_id = cursor.fetchone()[0]

                # Sensores Morfossilógicos
                if t.pos_ in counts: counts[t.pos_] += 1
                if t.morph.get("Person") == ["1"]: n_primeira_pessoa += 1
                if t.pos_ == 'NUM': tem_numeral = 1

                # Sensor de Identidade (Ontologia: 'ser'/'estar' como raiz ou cópula)
                if t.lemma_.lower() in ['ser', 'estar'] and (t.dep_ in ['ROOT', 'cop']):
                    is_identidade = 1

                if not t.is_stop and not t.is_punct:
                    palavras_sig_len.append(len(t.text))

                # Índice Invertido com Hierarquia
                cursor.execute("""
                    INSERT INTO verso_palavra (
                        verso_id, palavra_id, posicao, head_pos, dep_relation, morph
                    ) VALUES (?, ?, ?, ?, ?, ?)
                """, (verso_id, palavra_id, t.i, t.head.i, t.dep_, str(t.morph)))

            # --- MÉTRICAS DE NÍVEL MACRO ---
            score_emocional = (counts['ADJ'] + counts['ADV']) / total_tokens if total_tokens > 0 else 0
            score_informativo = (counts['PROPN'] + counts['NUM']) / total_tokens if total_tokens > 0 else 0
            score_acao = counts['VERB'] / total_tokens if total_tokens > 0 else 0
            avg_word_len = np.mean(palavras_sig_len) if palavras_sig_len else 0.0

            present_tags = [v for v in counts.values() if v > 0]
            entropia = -sum([(v/total_tokens) * np.log(v/total_tokens + 1e-9) for v in present_tags])

            # Texto Lematizado (Melhor para Zero-Shot)
            texto_limpo = " ".join([t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct])

            # Persistência dos Sensores XAI
            cursor.execute("""
                INSERT INTO verso_limpo (
                    verso_id, texto_limpo, score_emocional, score_informativo,
                    score_acao, entropia_gramatical, n_primeira_pessoa,
                    avg_word_len, is_identidade, tem_numeral
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (verso_id, texto_limpo, score_emocional, score_informativo,
                  score_acao, entropia, n_primeira_pessoa, avg_word_len,
                  is_identidade, tem_numeral))

        conn.commit()
        print("✅ Sucesso: Banco de dados atualizado e persistido.")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro de Bloqueio (Lock): {e}")
        if conn: conn.rollback()
    except Exception as e:
        print(f"❌ Erro inesperado: {e}")
        if conn: conn.rollback()
    finally:
        if conn:
            conn.close()
            print("🔒 Conexão fechada com segurança.")

# Execução (Certifique-se de que DB_PATH esteja definido)
executar_indexacao_completa(DB_PATH)

🧹 Limpando índices anteriores...
🧠 Analisando 1070 versos...


Gramática e Sensores:   0%|          | 0/1070 [00:00<?, ?it/s]

✅ Sucesso: Banco de dados atualizado e persistido.
🔒 Conexão fechada com segurança.


In [ ]:
# Célula 5: Classificação por Eixos Existenciais (IA + Regras de Negócio Literárias)
import sqlite3
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados conforme Modelo de Dados
conn = sqlite3.connect(DB_PATH)

# Recupera Eixos e as Sentenças de Descrição Qualificadas
query_eixos = """
    SELECT e.id, e.nome, GROUP_CONCAT(ed.sentenca, ' ') as descricao_completa
    FROM eixo e
    JOIN eixo_descricao ed ON e.id = ed.eixo_id
    GROUP BY e.id ORDER BY e.id
"""
df_eixos = pd.read_sql_query(query_eixos, conn)
labels_ia = df_eixos['descricao_completa'].tolist()

# Query integrada: Versos + Metadados XAI + Gênero Literário
query_input = """
    SELECT
        vl.*,
        gl.nome as genero_nome
    FROM verso_limpo vl
    JOIN verso v ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario gl ON l.genero_id = gl.id
    WHERE v.processar = 'S'
"""
df_input = pd.read_sql_query(query_input, conn)

# 2. Inicialização do Modelo BART (Zero-Shot Classification)
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0)

def classificar_hibrido(row, labels):
    # --- DNA SINTÁTICO (XAI) ---
    dna = []
    if row['n_primeira_pessoa'] > 0: dna.append("Relato Pessoal/Subjetivo")
    if row['is_identidade'] == 1: dna.append("Definição de Identidade/Estado")
    if row['tem_numeral'] == 1: dna.append("Dados Quantitativos/Inventário")

    prefixo = f"[Contexto: {', '.join(dna)}] " if dna else ""
    texto_para_ia = prefixo + row['texto_limpo']

    # Inferência da IA
    res = classifier(texto_para_ia, labels, multi_label=False)
    probs = [res['scores'][res['labels'].index(l)] for l in labels]

    # --- REGRAS DE SOBERANIA LITERÁRIA[cite: 1] ---
    genero = row['genero_nome']

    # Caso Jó: Poético/Sapiencial com Prólogo Narrativo[cite: 1]
    if genero == 'Poético/Sapiencial':
        # Se houver alta densidade de dados (nomes/números), prioriza Narrativo (Eixo 3)
        if row['score_informativo'] > 0.12 or row['tem_numeral'] == 1:
            probs[3] += 0.45
        else:
            # No corpo poético, penaliza o narrativo para focar na dialética existencial
            probs[3] -= 0.15

    # Caso Filipenses: Epístola[cite: 1]
    elif genero == 'Epístola':
        probs[3] -= 0.25 # Cartas raramente são puramente narrativas

    # Re-normalização (Soma = 1.0)
    probs = list(np.clip(probs, 0.001, 1.0))
    probs = [p / sum(probs) for p in probs]

    return probs

# 3. Processamento em Lote
results = []
print(f"🤖 Classificando {len(df_input)} versos (Jó e Fp)...")

for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Eixos"):
    probs = classificar_hibrido(row, labels_ia)

    # Métricas de Governança e Decisão
    idx_vencedor = np.argmax(probs)
    gap = sorted(probs, reverse=True)[0] - sorted(probs, reverse=True)[1]
    entropia = -sum([p * np.log(p + 1e-9) for p in probs])

    # Status de Decisão para o TCC[cite: 1]
    status = "IA + DNA Sintático"
    if gap > 0.45: status = "Alta Confiança"
    if row['genero_nome'] == 'Poético/Sapiencial' and idx_vencedor == 3:
        status = "Narrativa de Moldura (XAI)"
    elif entropia > 1.1:
        status = "Ambiguidade Poética"

    results.append({
        'verso_id': int(row['verso_id']),
        'topico_id': int(idx_vencedor),
        'p_exaustao': probs[0],
        'p_transitoriedade': probs[1],
        'p_vazio': probs[2],
        'p_narrativo': probs[3],
        'similaridade_final': probs[idx_vencedor],
        'margem_dominancia': gap,
        'status_decisao': status,
        'entropia': entropia,
        'gap_confianca': gap
    })

# 4. Persistência dos Resultados[cite: 1]
df_final = pd.DataFrame(results)
cursor = conn.cursor()
cursor.execute("DELETE FROM verso_topico")
df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("✨ Processamento concluído!")

In [ ]:
# Célula 6: Análise de Sentimento Contextualizada com DNA Sintático
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
import numpy as np
from tqdm.auto import tqdm

# 1. Inicializar o Analisador (BERTimbau-based para PT-BR)
print("🚀 Carregando modelo Transformer para Sentimento...")
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca de dados cruzados (Texto + Metadados XAI das Células 4 e 5)
conn = sqlite3.connect(DB_PATH)
query_cruzada = """
    SELECT
        v.id as verso_id,
        v.texto,
        vt.topico_id,
        vl.n_primeira_pessoa,
        vl.is_identidade
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    WHERE v.processar = 'S'
"""
df_input = pd.read_sql_query(query_cruzada, conn)

# 3. Execução da análise em lotes (Eficiência em GPU/CPU)
print(f"📊 Analisando carga emocional de {len(df_input)} versículos...")
textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        v_id = ids_lote[idx]

        # Localiza metadados do verso para o cálculo de explicabilidade
        meta = df_input[df_input['verso_id'] == v_id].iloc[0]

        # Ajuste de Peso: Relatos em 1ª pessoa ganham 50% mais peso na polaridade
        multiplicador_voz = 1.5 if meta['n_primeira_pessoa'] > 0 else 1.0
        sent_base = mapa_num.get(p.output, 0)

        sentimentos.append({
            'verso_id': int(v_id),
            'label': p.output,
            'sentimento_num': sent_base,
            'sentimento_ajustado': float(sent_base * multiplicador_voz),
            'score_pos': float(p.probas.get('POS', 0)),
            'score_neg': float(p.probas.get('NEG', 0)),
            'score_neu': float(p.probas.get('NEU', 0))
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados com Limpeza Prévia
try:
    cursor = conn.cursor()

    # LIMPEZA: Remove dados anteriores para evitar duplicidade ou lixo
    print("🧹 Limpando dados anteriores da tabela 'verso_sentimento'...")
    cursor.execute("DELETE FROM verso_sentimento")

    # INSERÇÃO: Adiciona os novos dados preservando a estrutura da tabela
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()

    print("\n✅ Célula 6 concluída! Sentimentos processados e integrados.")

    # 5. DIAGNÓSTICO FINAL: CRUZAMENTO EXISTENCIAL
    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            -- Cura: Identidade + Sentimento Positivo
            SUM(CASE WHEN vs.sentimento_num = 1 AND vl.is_identidade = 1 THEN 1 ELSE 0 END) as Definicoes_Fortalecedoras,
            -- Crise: 1ª Pessoa + Sentimento Negativo
            SUM(CASE WHEN vs.sentimento_num = -1 AND vl.n_primeira_pessoa > 0 THEN 1 ELSE 0 END) as Lamentos_Pessoais,
            ROUND(AVG(vs.sentimento_ajustado), 3) as Polaridade_Ponderada
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso_limpo vl ON vt.verso_id = vl.verso_id
        WHERE t.id != 3
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Ponderada DESC
    """, conn)

    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
    conn.rollback()
finally:
    conn.close()